# Violence Model Training (Kaggle)

Trains **1 model**: `VIOLENCE` vs `NORMAL`.

**Model:** EfficientNet-B0 (pretrained) + average over sampled frames.
- Strong accuracy for the complexity
- Fits free Kaggle GPUs
- No ViT / Mamba / fancy research models

**How to run:**
1. Open this notebook on Kaggle (GPU on)
2. Enable Internet in notebook settings
3. Run all cells
4. Download `violence_best.pt` from Output
5. Put it in your project `models/violence_best.pt`

In [ ]:
# Install deps (Kaggle already has torch/torchvision usually)
%pip install -q kagglehub opencv-python-headless tqdm scikit-learn

In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path

import cv2
import kagglehub
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from tqdm.auto import tqdm

# -------------------- CONFIG --------------------
SEED = 42
NUM_FRAMES = 8          # frames per clip
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 8
LR = 3e-4
NUM_WORKERS = 2
VAL_RATIO = 0.2
MAX_VIDEOS_PER_CLASS = None  # set e.g. 300 for a faster smoke run

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = OUT_DIR / "violence_best.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Reduce OpenCV/ffmpeg decode spam from a few broken videos
try:
    cv2.setLogLevel(0)
except Exception:
    pass

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Download dataset
data_root = Path(kagglehub.dataset_download("magicearth25/video-violence-detection-dataset"))
print("Dataset path:", data_root)

# Show top-level layout so we can see class folders
for p in sorted(data_root.rglob("*"))[:40]:
    print(p.relative_to(data_root))

In [ ]:
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

# This dataset layout:
#   .../RLVS/train/Fight/*.mp4
#   .../RLVS/train/NonFight/*.mp4
#   .../RLVS/test/Fight/*.mp4
#   .../RLVS/test/NonFight/*.mp4
#
# IMPORTANT: do NOT search the full path for "fight"/"violence".
# The root folder is named "Violence Fight Detection dataset", which would
# wrongly label every video as VIOLENCE.


def normalize_name(name: str) -> str:
    return name.lower().replace("_", "").replace("-", "").replace(" ", "")


POSITIVE_FOLDER_NAMES = {"fight", "violence", "violent", "fighting"}
NEGATIVE_FOLDER_NAMES = {"nonfight", "nonviolence", "nonviolent", "normal"}


def infer_label_from_path(path: Path) -> int | None:
    # Use only the immediate parent folder of the video file
    parent = normalize_name(path.parent.name)
    if parent in NEGATIVE_FOLDER_NAMES:
        return 0  # NORMAL
    if parent in POSITIVE_FOLDER_NAMES:
        return 1  # VIOLENCE
    return None


def collect_videos(root: Path):
    items = []
    for path in root.rglob("*"):
        if path.suffix.lower() not in VIDEO_EXTS:
            continue
        label = infer_label_from_path(path)
        if label is None:
            continue
        items.append((path, label))
    return items


all_items = collect_videos(data_root)
print("Labeled videos found:", len(all_items))
print("VIOLENCE:", sum(1 for _, y in all_items if y == 1))
print("NORMAL:", sum(1 for _, y in all_items if y == 0))

if len(all_items) == 0:
    raise RuntimeError(
        "Could not auto-detect class folders. Expected parent folders named Fight / NonFight."
    )

if len({y for _, y in all_items}) < 2:
    raise RuntimeError(
        "Only one class found. Check that both Fight and NonFight folders exist."
    )

if MAX_VIDEOS_PER_CLASS is not None:
    by_class = {0: [], 1: []}
    for item in all_items:
        by_class[item[1]].append(item)
    all_items = []
    for cls, rows in by_class.items():
        random.shuffle(rows)
        all_items.extend(rows[:MAX_VIDEOS_PER_CLASS])
    print("After capping:", len(all_items))

paths = [p for p, _ in all_items]
labels = [y for _, y in all_items]

train_paths, val_paths, train_y, val_y = train_test_split(
    paths, labels, test_size=VAL_RATIO, random_state=SEED, stratify=labels
)
print("Train:", len(train_paths), "Val:", len(val_paths))


In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT
preprocess = weights.transforms()


def sample_frames(video_path: Path, num_frames: int = NUM_FRAMES) -> list[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    frames = []

    if total <= 0:
        # Fallback: read sequentially
        while len(frames) < num_frames:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    else:
        idxs = np.linspace(0, max(total - 1, 0), num_frames).astype(int)
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if not ok:
                continue
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    cap.release()

    if not frames:
        raise RuntimeError(f"No frames read from: {video_path}")

    # Pad by repeating last frame if needed
    while len(frames) < num_frames:
        frames.append(frames[-1])

    return frames[:num_frames]


class VideoClipDataset(Dataset):
    def __init__(self, paths, labels, train: bool):
        self.paths = list(paths)
        self.labels = list(labels)
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = Path(self.paths[idx])
        y = int(self.labels[idx])
        try:
            frames = sample_frames(path)
        except Exception as e:
            # Rare broken video: return zeros so training continues
            print(f"Skip broken video {path}: {e}")
            x = torch.zeros(NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE)
            return x, y

        tensors = []
        for frame in frames:
            # torchvision preprocess expects PIL or tensor; use tensor HxWxC -> CxHxW float later via transforms
            from PIL import Image

            img = Image.fromarray(frame)
            tensors.append(preprocess(img))

        x = torch.stack(tensors, dim=0)  # T,C,H,W
        return x, y


train_ds = VideoClipDataset(train_paths, train_y, train=True)
val_ds = VideoClipDataset(val_paths, val_y, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("Loaders ready")

In [ ]:
class ClipClassifier(nn.Module):
    """EfficientNet-B0 over frames, then mean-pool in time."""

    def __init__(self, num_classes: int = 2):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x: B,T,C,H,W
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1).mean(dim=1)
        return self.head(feats)


model = ClipClassifier().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

print(model)

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        loss_sum += float(loss.item()) * y.size(0)
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.size(0))
    return loss_sum / max(total, 1), correct / max(total, 1)


best_acc = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    seen = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for x, y in pbar:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += float(loss.item()) * y.size(0)
        seen += int(y.size(0))
        pbar.set_postfix(loss=running / max(seen, 1))

    train_loss = running / max(seen, 1)
    val_loss, val_acc = evaluate(val_loader)
    history.append((epoch, train_loss, val_loss, val_acc))
    print(f"epoch={epoch} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(
            {
                "model_name": "violence",
                "arch": "efficientnet_b0_temporal_mean",
                "num_frames": NUM_FRAMES,
                "img_size": IMG_SIZE,
                "label_map": {"NORMAL": 0, "VIOLENCE": 1},
                "val_acc": best_acc,
                "state_dict": model.state_dict(),
            },
            BEST_PATH,
        )
        print("Saved best ->", BEST_PATH, "acc=", best_acc)

print("Best val accuracy:", best_acc)
print("Download this file from Kaggle Output:", BEST_PATH)

## After training

1. Download `violence_best.pt`
2. Copy into your project:
   `models/violence_best.pt`
3. In `.env`:
   ```
   VIOLENCE_MODEL_WEIGHTS_PATH=models/violence_best.pt
   ```
4. Then implement / enable loading in `src/cctv_ai/inference/violence/model_adapter.py`